# 09 - Practical Variational Inference Tips and Diagnostics

By now, the main ideas of VI are in place.
This notebook focuses on practical questions:
- How do we know optimization is behaving well?
- How do we detect underfitting in the variational family?
- What common failure modes should we watch for?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## Part 1: Monitor the ELBO

The ELBO is the main training objective.
A healthy run often shows an ELBO that trends upward, though it can be noisy in stochastic optimization.

In [ ]:
steps = np.arange(200)
trend = -120 + 25 * (1 - np.exp(-steps / 50))
noise = np.random.normal(scale=0.8, size=len(steps))
elbo_trace = trend + noise

plt.figure(figsize=(8, 4))
plt.plot(steps, elbo_trace, color='navy', linewidth=2)
plt.title('Example ELBO curve')
plt.xlabel('iteration')
plt.ylabel('ELBO')
plt.show()

## Part 2: Watch the KL term and reconstruction term separately

In many latent-variable models, the ELBO has two main pieces:
- reconstruction quality
- KL regularization

Tracking them separately often reveals issues hidden by the total ELBO.

In [ ]:
reconstruction = -80 + 18 * (1 - np.exp(-steps / 40)) + np.random.normal(scale=0.7, size=len(steps))
kl = 25 - 8 * (1 - np.exp(-steps / 70)) + np.random.normal(scale=0.4, size=len(steps))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(steps, reconstruction, color='seagreen', linewidth=2)
axes[0].set_title('Reconstruction term')
axes[0].set_xlabel('iteration')

axes[1].plot(steps, kl, color='crimson', linewidth=2)
axes[1].set_title('KL term')
axes[1].set_xlabel('iteration')

plt.tight_layout()
plt.show()

## Part 3: Common failure modes

### 1. Under-expressive variational family
If $q$ is too simple, optimization may succeed but the approximation can still be poor.

### 2. Posterior collapse
In VAEs, the encoder may ignore the latent variable and the KL term may go near zero.

### 3. Unstable optimization
The ELBO may oscillate wildly if the learning rate is too high or gradients are too noisy.

In [ ]:
# Simulated posterior collapse style signal
kl_collapse = 4.0 * np.exp(-steps / 18) + 0.05 * np.random.normal(size=len(steps))

plt.figure(figsize=(8, 4))
plt.plot(steps, kl_collapse, color='crimson', linewidth=2)
plt.title('Example warning sign: KL collapsing toward zero')
plt.xlabel('iteration')
plt.ylabel('KL term')
plt.show()

## Part 4: Practical checklist

When training a VI model, check:
1. Is the ELBO improving overall?
2. Are reconstruction and KL both behaving sensibly?
3. Does the learned posterior show enough variability?
4. Does a richer variational family help?
5. Are results stable across random seeds?

## Summary

What to remember:
1. The ELBO is necessary but not always sufficient as a diagnostic
2. Separate monitoring of reconstruction and KL is very useful
3. Simple variational families can hide approximation bias
4. Practical VI requires both theory and careful monitoring

In [ ]:
# Exercises
# 1) Simulate a more unstable ELBO trace by increasing the noise.
# 2) Sketch what a healthy KL trace might look like in a model you know.
# 3) Think about when richer variational families are worth the extra cost.

pass